In [0]:
%run ./_utils

### Export `deleted_ids.csv.gz` (oxjob #784)

Writes the cumulative deleted-works ledger (`openalex.works.deleted_works`,
maintained nightly by `notebooks/end2end/TrackDeletedWorks`) to
`full/{date}/{format}/works/deleted_ids.csv.gz` (gzip-compressed CSV) — inside BOTH format trees'
works directory (like `manifest.json`) so the entity is named by the path and
the quarterly `sync_to_public` copy picks it up with the rest of `works/`.
One row per deleted work: `work_id` (`https://openalex.org/W…`, same id form as
the works entity files) and `deleted_date` (the date the work disappeared from
`openalex_works`; for works ledgered by backfill sweeps this is the detection
date, not the true deletion date). Works that reappear are removed from the
ledger, so a live work never shows up here — the file can shrink between days.
The uncompressed `deleted_ids.csv` is no longer written (removed 2026-09-24; the
2.1 GB plain file duplicated the 160 MB gzip). The gzip file is written even when the
ledger is empty or absent (header only). Consumers apply it as: remove these ids from
any locally-held copy of works.

Task ordering: depends on `export_works` because that export `rm -r`s the
`{format}/works/` directories before writing — this file must land after.


In [0]:
LEDGER = "openalex.works.deleted_works"

date_str = get_snapshot_date()
out_dirs = [f"{S3_BASE}/{date_str}/{fmt}/works" for fmt in ("jsonl", "parquet")]
tmp_dir = f"{S3_BASE}/{date_str}/_temp/deleted_csv"

ledger_exists = spark.catalog.tableExists(LEDGER)
if ledger_exists:
    df = (
        spark.table(LEDGER)
        .selectExpr("CONCAT('https://openalex.org/W', work_id) AS work_id", "deleted_date")
        .distinct()
        .coalesce(1)
        .sortWithinPartitions("deleted_date", "work_id")
    )
else:
    # Use the same writer so the header-only fallback is also valid gzip.
    df = spark.createDataFrame([], "work_id STRING, deleted_date DATE").coalesce(1)

def single_part(path, suffix):
    parts = [f.path for f in dbutils.fs.ls(path)
             if f.name.startswith("part-") and f.name.endswith(suffix)]
    if len(parts) != 1:
        raise RuntimeError(f"Expected one deletion-log part ending in {suffix}, found {len(parts)}")
    return parts[0]


# Write the gzip-compressed CSV directly; the uncompressed deleted_ids.csv is no longer produced (2026-09-24).
gzip_dir = f"{tmp_dir}/gzip"
(df.write.mode("overwrite")
    .option("header", True)
    .option("compression", "gzip")
    .csv(gzip_dir))
gzip_part = single_part(gzip_dir, ".csv.gz")

out_paths = []
for out_dir in out_dirs:
    out_path = f"{out_dir}/deleted_ids.csv.gz"
    if not dbutils.fs.cp(gzip_part, out_path):
        raise RuntimeError(f"Failed to copy deletion log to {out_path}")
    out_paths.append(out_path)
dbutils.fs.rm(tmp_dir, recurse=True)

if ledger_exists:
    count = spark.table(LEDGER).select("work_id").distinct().count()
    print(f"Wrote {count:,} deleted works to:")
else:
    print(f"{LEDGER} does not exist yet; wrote header-only files to:")

for out_path in out_paths:
    print(f"  {out_path}")

# One-time cleanup: 2026-08-15 shipped this file at the snapshot root before the
# works-directory placement landed. Harmless no-op on any other date.
try:
    dbutils.fs.rm(f"{S3_BASE}/{date_str}/deleted_ids.csv")
    print(f"Removed legacy root-level deleted_ids.csv from {date_str}")
except Exception:
    pass
